In [53]:
import numpy as np 
import pandas as pd
import seaborn 
import matplotlib.pyplot as plt
from LinearRegression import GradientDescent



# doc dataset

In [54]:
df = pd.read_csv("vietnam_housing_dataset.csv")
print("Danh sách các cột trong dataset:", df.columns.tolist())

Danh sách các cột trong dataset: ['Address', 'Area', 'Frontage', 'Access Road', 'House direction', 'Balcony direction', 'Floors', 'Bedrooms', 'Bathrooms', 'Legal status', 'Furniture state', 'Price']


# Buoc 1:chia tap train va test 

### xao tron chi so thu cong

In [55]:
np.random.seed(42)
indices = np.arange(len(df))
np.random.shuffle(indices)

### tach thanh 2 tap ( 80 : 20)

In [56]:
test_size = int(len(df) * 0.2)
test_indices = indices[:test_size]
train_indices = indices[test_size:]

df_train = df.iloc[train_indices].copy()
df_test = df.iloc[test_indices].copy()

print(f"Tổng số mẫu dữ liệu : {len(df)}")
print(f"Tập Train (80%)     : {len(df_train)} dòng")
print(f"Tập Test (20%)      : {len(df_test)} dòng")

Tổng số mẫu dữ liệu : 30229
Tập Train (80%)     : 24184 dòng
Tập Test (20%)      : 6045 dòng


# Buoc2: XU LY DU LIEU

### lam sach cot price 

In [57]:
df_train_clean = df_train.copy()
df_train_clean['Price'] = pd.to_numeric(df_train_clean['Price'], errors='coerce')
df_train_clean['Area'] = pd.to_numeric(df_train_clean['Area'], errors='coerce')
df_train_clean = df_train_clean.dropna(subset=['Price', 'Area'])

## xy ly cot address, tach lay quan huyen

In [58]:
df_train_clean['District'] = df_train_clean['Address'].astype(str).apply(
    lambda x: x.split(',')[-2].strip() if len(x.split(',')) >= 2 else 'Other'
)

## xu ly cac cot dang so 

In [59]:
num_cols = ['Area', 'Frontage', 'Access Road', 'Floors', 'Bedrooms', 'Bathrooms']
medians = {}
for col in num_cols:
    if col in df_train_clean.columns:
        df_train_clean[col] = pd.to_numeric(df_train_clean[col], errors='coerce')
        medians[col] = df_train_clean[col].median()
        df_train_clean[col] = df_train_clean[col].fillna(medians[col])

## ma hoa cac cot dang chu ( category to number ), su dung onehot-coding

In [60]:
cat_cols = ['House direction', 'Balcony direction', 'Legal status', 'Furniture state', 'District']
cat_cols_exist = [c for c in cat_cols if c in df_train_clean.columns]
df_train_clean = pd.get_dummies(df_train_clean, columns=cat_cols_exist, drop_first=True)
if 'Address' in df_train_clean.columns:
    df_train_clean = df_train_clean.drop(columns=['Address'])

### Xu ly tap test

In [61]:

df_test_clean = df_test.copy()
df_test_clean['Price'] = pd.to_numeric(df_test_clean['Price'], errors='coerce')
df_test_clean['Area'] = pd.to_numeric(df_test_clean['Area'], errors='coerce')
df_test_clean = df_test_clean.dropna(subset=['Price', 'Area'])

df_test_clean['District'] = df_test_clean['Address'].astype(str).apply(
    lambda x: x.split(',')[-2].strip() if len(x.split(',')) >= 2 else 'Other'
)

for col in num_cols:
    if col in df_test_clean.columns:
        df_test_clean[col] = pd.to_numeric(df_test_clean[col], errors='coerce')
        df_test_clean[col] = df_test_clean[col].fillna(medians[col])

df_test_clean = pd.get_dummies(df_test_clean, columns=cat_cols_exist, drop_first=True)
if 'Address' in df_test_clean.columns:
    df_test_clean = df_test_clean.drop(columns=['Address'])

# Căn chỉnh cột tập Test khớp 100% với tập Train
df_test_clean = df_test_clean.reindex(columns=df_train_clean.columns, fill_value=0)

# Tách ra ma trận X và vector y
X_train = df_train_clean.drop(columns=['Price']).values.astype(float)
y_train = df_train_clean['Price'].values

X_test = df_test_clean.drop(columns=['Price']).values.astype(float)
y_test = df_test_clean['Price'].values


### Ham chuan hoa du lieu (Feature Scaling / Standard Scaler)

In [62]:
# Tính Mean và Std trên tập Train
train_mean = np.mean(X_train, axis=0)
train_std = np.std(X_train, axis=0) + 1e-8

# Scale cả Train và Test bằng Mean, Std của Train
X_train_scaled = (X_train - train_mean) / train_std
X_test_scaled = (X_test - train_mean) / train_std

# BƯỚC 3: Đánh giá 5 Folds trên tập Test

Khoi tao va huan luyen mo hinh


In [63]:

model = GradientDescent.LinearRegressionGD(lr=0.01, epochs=1000)
model.fit(X_train_scaled, y_train_log)

AttributeError: module 'LinearRegression.GradientDescent' has no attribute 'LinearRegressionGD'

In [ ]:
def compute_mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

# Xáo trộn và chia 5 Folds tập Test (20%)
np.random.seed(42)
test_len = len(X_test_scaled)
test_idx = np.arange(test_len)
np.random.shuffle(test_idx)

folds_idx = np.array_split(test_idx, 5)
mse_folds = []

print("--- ĐÁNH GIÁ MÔ HÌNH TRÊN 5 FOLDS CỦA TẬP TEST ---")

for i, f_idx in enumerate(folds_idx):
    X_fold = X_test_scaled[f_idx]
    y_fold = y_test[f_idx]

    # Dự đoán bằng hàm predict() của Class
    y_pred_fold = model.predict(X_fold)
    
    fold_mse = compute_mse(y_fold, y_pred_fold)
    mse_folds.append(fold_mse)

    print(f"Fold {i+1} ({len(f_idx)} dòng test): MSE = {fold_mse:.4f}")

print("\n================ TỔNG HỢP KẾT QUẢ TEST ================")
print(f"MSE Trung bình      : {np.mean(mse_folds):.4f}")
print(f"Độ lệch chuẩn (std) : ±{np.std(mse_folds):.4f}")

--- ĐÁNH GIÁ MÔ HÌNH TRÊN 5 FOLDS CỦA TẬP TEST ---


NameError: name 'model' is not defined